In [1]:
import pandas as pd
import glob
import os

In [5]:
path = r'C:\Users\lidon\Desktop\2026spring\IDX_MLS_Analytics'

# 1. Vertically concatenate all Listing files
listing_files = glob.glob(os.path.join(path, "CRMLSListing*.csv"))
all_listings = pd.concat((pd.read_csv(f, low_memory=False) for f in listing_files), ignore_index=True)

# 2. Vertically concatenate Sold files (prioritizing '_filled' versions)
# Retrieve all available sold files first
all_sold_files = glob.glob(os.path.join(path, "CRMLSSold*.csv"))

# Filter and extract the correct source file paths
final_sold_paths = []
months = set([os.path.basename(f)[9:15] for f in all_sold_files]) # Extract YYYYMM format

for month in sorted(months):
    # Check if a pre-filled version exists for the current month
    filled_file = os.path.join(path, f"CRMLSSold{month}_filled.csv")
    raw_file = os.path.join(path, f"CRMLSSold{month}.csv")
    
    if os.path.exists(filled_file):
        final_sold_paths.append(filled_file)  # Prioritize the filled version
    elif os.path.exists(raw_file):
        final_sold_paths.append(raw_file)    # Fallback to the raw version if filled is unavailable

# Concatenate the filtered monthly data into a master Sold historical dataframe
all_sold = pd.concat((pd.read_csv(f, low_memory=False) for f in final_sold_paths), ignore_index=True)

# ==================== Data Volume Statistics ====================
print("================ IDX Data File Statistics ================")
print(f"1. Total monthly Listing files scanned: {len(listing_files)}")
print(f"2. Total raw Sold files detected (including duplicates): {len(all_sold_files)}")
print(f"3. Final deduplicated Sold files utilized (applying '_filled' priority): {len(final_sold_paths)}")
print("==========================================================")

print(f"➔ Total records in merged Listings master dataframe: {all_listings.shape[0]} rows")
print(f"➔ Total records in merged Sold master dataframe: {all_sold.shape[0]} rows")

================ IDX Data File Statistics ================
1. Total monthly Listing files scanned: 27
2. Total raw Sold files detected (including duplicates): 31
3. Final deduplicated Sold files utilized (applying '_filled' priority): 28
➔ Total records in merged Listings master dataframe: 866140 rows
➔ Total records in merged Sold master dataframe: 615707 rows
